# 5-Step DuPont Decomposition — Greek Systemic Banks (2022–2024)

**Methodology:** Banking-adapted 5-step DuPont (Petersen & Plenborg, *Financial Statement Analysis*, 2012).
Standard corporate DuPont uses EBIT, which is undefined for banks. This notebook uses the banking equivalent:

$$\text{ROE} = \underbrace{\frac{\text{NP}}{\text{PBT}}}_{\text{Tax Burden}} \times \underbrace{\frac{\text{PBT}}{\text{PPOP}}}_{\text{Provision Burden}} \times \underbrace{\frac{\text{PPOP}}{\text{Op. Income}}}_{\text{Pre-provision Margin}} \times \underbrace{\frac{\text{Op. Income}}{\text{Assets}}}_{\text{Asset Utilisation}} \times \underbrace{\frac{\text{Assets}}{\text{Equity}}}_{\text{Leverage}}$$

where **PPOP** = Pre-Provision Operating Profit = PBT + |Impairment losses|.

**Sources:** Official annual reports (PDFs in `02_Banking_Sector_Dashboard/data/raw/`).  
**Data verified:** All figures cross-checked against source PDF pages (see `DATA_DICTIONARY.md`).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

# ── Brand colors (bank logo palette) ──────────────────────────────────────────
COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS  = list(COLORS.keys())
YEARS  = [2022, 2023, 2024]

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')

print('Libraries loaded. Data directory:', DATA_DIR.resolve())

In [ ]:
# ── Load processed data ───────────────────────────────────────────────────────
kpis = pd.read_csv(DATA_DIR / 'kpis_final.csv')
is_long = pd.read_csv(DATA_DIR / 'income_statement_final.csv')

# Pivot income statement to extract Profit Before Tax (not in kpis_final)
is_pivot = (
    is_long
    .pivot_table(index=['bank', 'year'], columns='metric', values='value', aggfunc='first')
    .reset_index()
)
pbt_col = 'Profit before tax'
assert pbt_col in is_pivot.columns, f"Column '{pbt_col}' not found in income statement."

# Merge PBT into kpis
df = kpis.merge(
    is_pivot[['bank', 'year', pbt_col]].rename(columns={pbt_col: 'profit_before_tax'}),
    on=['bank', 'year'],
    how='left'
)

print(f'Loaded {len(df)} bank-year observations. Columns: {list(df.columns)}')
df[['bank', 'year', 'net_profit', 'profit_before_tax', 'impairment', 'operating_income', 'total_assets', 'equity', 'roe']].round(1)

In [ ]:
# ── Compute 5-step DuPont components ─────────────────────────────────────────
#
# PPOP = Profit Before Tax + |Impairment losses|
# (impairment column in kpis is negative, so we subtract to add back)
#
df['ppop'] = df['profit_before_tax'] + df['impairment'].abs()

df['tax_burden']         = df['net_profit']       / df['profit_before_tax']  # (1) NP / PBT
df['provision_burden']   = df['profit_before_tax'] / df['ppop']              # (2) PBT / PPOP
df['preprovision_margin']= df['ppop']              / df['operating_income']   # (3) PPOP / OpInc
df['asset_utilisation']  = df['operating_income']  / df['total_assets']      # (4) OpInc / Assets
df['leverage']           = df['total_assets']      / df['equity']             # (5) Assets / Equity

# Reconstruct ROE and express as %
df['roe_reconstructed'] = (
    df['tax_burden']
    * df['provision_burden']
    * df['preprovision_margin']
    * df['asset_utilisation']
    * df['leverage']
    * 100
)

components = ['tax_burden', 'provision_burden', 'preprovision_margin', 'asset_utilisation', 'leverage']
print('DuPont components computed.')
df[['bank', 'year'] + components + ['roe_reconstructed', 'roe']].round(4)

In [ ]:
# ── Sanity check: reconstructed ROE must match reported within ±0.5 pp ────────
df['roe_error_pp'] = (df['roe_reconstructed'] - df['roe']).abs()

MAX_TOLERANCE_PP = 0.5
failures = df[df['roe_error_pp'] > MAX_TOLERANCE_PP][['bank', 'year', 'roe', 'roe_reconstructed', 'roe_error_pp']]

print('\n── ROE Reconstruction Check (' + f'tolerance ±{MAX_TOLERANCE_PP}pp) ──')
print(df[['bank', 'year', 'roe', 'roe_reconstructed', 'roe_error_pp']].to_string(index=False))

if len(failures) > 0:
    print('\n⚠️  FAILURES:')
    print(failures.to_string(index=False))
    raise AssertionError(f'{len(failures)} bank-year(s) exceed ±{MAX_TOLERANCE_PP}pp tolerance. Investigate data.')
else:
    print(f'\n✅ All 12 bank-years reconstruct within ±{MAX_TOLERANCE_PP}pp.')

In [ ]:
# ── Chart 1: Small Multiples — Component Evolution Per Bank ──────────────────
# For each bank: how did each DuPont lever change from 2022 → 2024?
# Normalised to 2022 = 100 so all 5 levers are comparable on one axis.

COMP_LABELS = {
    'tax_burden':          'Tax Burden (NP/PBT)',
    'provision_burden':    'Provision Burden (PBT/PPOP)',
    'preprovision_margin': 'Pre-provision Margin (PPOP/OpInc)',
    'asset_utilisation':   'Asset Utilisation (OpInc/Assets)',
    'leverage':            'Leverage (Assets/Equity)',
}
COMP_COLORS = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A']

fig1 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'<b>{b}</b>' for b in BANKS],
    shared_xaxes=False,
    vertical_spacing=0.14,
    horizontal_spacing=0.10,
)

for bi, bank in enumerate(BANKS):
    row, col = divmod(bi, 2)
    bank_df = df[df['bank'] == bank].sort_values('year')
    base = bank_df.set_index('year')[components].loc[2022]  # 2022 base

    for ci, (comp, color) in enumerate(zip(components, COMP_COLORS)):
        indexed = (bank_df[comp].values / base[comp]) * 100
        fig1.add_trace(
            go.Scatter(
                x=YEARS, y=indexed,
                name=COMP_LABELS[comp],
                line=dict(color=color, width=2),
                marker=dict(size=7),
                showlegend=(bi == 0),
                legendgroup=comp,
            ),
            row=row + 1, col=col + 1,
        )

    # Overlay reconstructed ROE trend (dashed)
    roe_indexed = (bank_df['roe_reconstructed'].values / bank_df['roe_reconstructed'].values[0]) * 100
    fig1.add_trace(
        go.Scatter(
            x=YEARS, y=roe_indexed,
            name='ROE (indexed)',
            line=dict(color='white', width=2.5, dash='dash'),
            marker=dict(size=8, symbol='diamond'),
            showlegend=(bi == 0),
            legendgroup='roe',
        ),
        row=row + 1, col=col + 1,
    )

    # Add 100 baseline
    fig1.add_hline(y=100, line_dash='dot', line_color='rgba(255,255,255,0.3)', row=row + 1, col=col + 1)

fig1.update_layout(
    title_text='<b>DuPont Component Evolution</b> | Indexed to 2022 = 100 | Greek Systemic Banks',
    title_font_size=16,
    template='plotly_dark',
    height=620,
    legend=dict(orientation='h', yanchor='bottom', y=-0.22, x=0.5, xanchor='center', font_size=11),
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
)
fig1.update_xaxes(tickvals=YEARS, ticktext=[str(y) for y in YEARS])
fig1.update_yaxes(title_text='Index (2022=100)', ticksuffix='')
fig1.show()

In [ ]:
# ── Chart 2: Cross-Bank Comparison Per DuPont Lever (2024) ───────────────────
# For 2024: which bank leads on each lever? This is the equity-research comparison.

df24 = df[df['year'] == 2024].set_index('bank')

# Use 2×3 layout (5 components + ROE)
fig2 = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Tax Burden (NP/PBT)',
        'Provision Burden (PBT/PPOP)',
        'Pre-provision Margin (PPOP/OpInc)',
        'Asset Utilisation (OpInc/Assets, %)',
        'Leverage (Assets/Equity, ×)',
        'ROE (%, reported)',
    ],
    vertical_spacing=0.18,
    horizontal_spacing=0.10,
)

metrics = components + ['roe']
positions = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]
multipliers = [1, 1, 1, 100, 1, 1]  # asset utilisation shown as %

for (metric, (r, c), mult) in zip(metrics, positions, multipliers):
    values = [df24.loc[b, metric] * mult for b in BANKS if b in df24.index]
    bar_colors = [COLORS[b] for b in BANKS if b in df24.index]
    fig2.add_trace(
        go.Bar(
            x=BANKS,
            y=values,
            marker_color=bar_colors,
            text=[f'{v:.2f}' if mult == 1 else f'{v:.2f}%' for v in values],
            textposition='outside',
            showlegend=False,
        ),
        row=r, col=c,
    )

fig2.update_layout(
    title_text='<b>DuPont Lever Comparison — 2024</b> | Greek Systemic Banks',
    title_font_size=16,
    template='plotly_dark',
    height=600,
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
)
fig2.update_xaxes(tickfont_size=11)
fig2.show()

In [ ]:
# ── Chart 3: ROE Bridge — What drove ROE differences in 2024? ────────────────
# Compare each bank vs sector median on each DuPont lever.
# Shows which bank is above / below peer median and by how much (in log-contribution).

df24_comp = df[df['year'] == 2024].copy()
sector_medians = df24_comp[components].median()

# Log-contribution relative to sector median (sum = ROE premium over median)
for comp in components:
    df24_comp[f'log_{comp}'] = np.log(df24_comp[comp] / sector_medians[comp])

log_cols = [f'log_{c}' for c in components]
df24_comp['total_log_premium'] = df24_comp[log_cols].sum(axis=1)

fig3 = go.Figure()
bar_width = 0.15
for bi, bank in enumerate(BANKS):
    row_b = df24_comp[df24_comp['bank'] == bank].iloc[0]
    log_vals = [row_b[f'log_{c}'] for c in components]
    x_pos = np.arange(len(components)) + bi * bar_width - (len(BANKS) - 1) * bar_width / 2
    fig3.add_trace(go.Bar(
        x=x_pos,
        y=log_vals,
        name=bank,
        marker_color=COLORS[bank],
        width=bar_width,
        text=[f'{v:+.3f}' for v in log_vals],
        textposition='outside',
        textfont_size=9,
    ))

fig3.add_hline(y=0, line_color='rgba(255,255,255,0.4)', line_dash='dash')
comp_abbrev = ['Tax Burden', 'Provision Burden', 'Pre-prov. Margin', 'Asset Util.', 'Leverage']
fig3.update_layout(
    title_text='<b>ROE Driver Analysis — 2024</b> | Log premium/discount vs sector median per DuPont lever',
    title_font_size=15,
    template='plotly_dark',
    height=430,
    xaxis=dict(
        tickmode='array',
        tickvals=np.arange(len(components)),
        ticktext=comp_abbrev,
    ),
    yaxis_title='Log premium vs sector median (+ = above median)',
    legend=dict(orientation='h', y=-0.20, x=0.5, xanchor='center'),
    barmode='group',
    paper_bgcolor='#0f1117',
    plot_bgcolor='#0f1117',
)
fig3.show()

In [ ]:
# ── Key Insights Table ────────────────────────────────────────────────────────
print('=' * 72)
print('  DuPont Summary — Greek Systemic Banks (2022–2024)')
print('=' * 72)

display_cols = [
    ('bank', 'Bank'), ('year', 'Year'),
    ('tax_burden', 'Tax Burden'), ('provision_burden', 'Prov. Burden'),
    ('preprovision_margin', 'PreProv. Margin'), ('asset_utilisation', 'Asset Util. (%)'),
    ('leverage', 'Leverage (×)'), ('roe', 'ROE (%)'),
]

out = df[['bank','year','tax_burden','provision_burden','preprovision_margin','asset_utilisation','leverage','roe']].copy()
out['asset_utilisation'] = (out['asset_utilisation'] * 100).round(2)
out[['tax_burden','provision_burden','preprovision_margin']] = out[['tax_burden','provision_burden','preprovision_margin']].round(3)
out['leverage'] = out['leverage'].round(2)
out['roe'] = out['roe'].round(1)

print(out.to_string(index=False))

print('\n── Key Findings ──')

# Find who has the highest/lowest of each component in 2024
for comp, label in zip(components, comp_abbrev):
    mult = 100 if comp == 'asset_utilisation' else 1
    best_bank = df24_comp.set_index('bank')[comp].idxmax()
    best_val  = df24_comp.set_index('bank')[comp].max() * mult
    worst_bank = df24_comp.set_index('bank')[comp].idxmin()
    worst_val  = df24_comp.set_index('bank')[comp].min() * mult
    unit = '%' if comp == 'asset_utilisation' else '×' if comp == 'leverage' else ''
    fmt = '.2f' if comp in ('asset_utilisation', 'leverage') else '.3f'
    print(f'  {label:<22}  Best: {best_bank} ({best_val:{fmt}}{unit})   '
          f'Worst: {worst_bank} ({worst_val:{fmt}}{unit})')

In [ ]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(df) == 12,              'Expected 12 bank-year rows (4 banks × 3 years).'
assert df['profit_before_tax'].notna().all(), 'Missing profit_before_tax values.'
assert df['roe_error_pp'].max() < 0.5,        'ROE reconstruction error exceeds ±0.5pp.'

# Component sanity: all multipliers should be positive
for comp in components:
    assert (df[comp] > 0).all(), f'Non-positive value in component: {comp}'

print('✅ All checks passed.')
print(f'   Max ROE reconstruction error: {df["roe_error_pp"].max():.4f} pp')
print(f'   Mean ROE reconstruction error: {df["roe_error_pp"].mean():.4f} pp')